# LMC2 #333, Colab çalışma sırası

Başlamadan önce eski notebook'un oturumunu kapat: Çalışma zamanı > Oturumları yönet > eski notebook > Sonlandır.

Sonra Çalışma zamanı > Tümünü çalıştır.

1. Kurulum ve yol kontrolü
2. Run A'nın resmi tarza benzer skoru ve süre ölçümü
3. Eğitim verisinin geri yüklenmesi (yaklaşık 17 dakika)
4. Base held-out eğitimine kaldığı yerden devam
5. Fokal düzlem testi
6. smp seed 123 ve 456 eğitimleri

Bağlantı koparsa yeniden bağlan ve yine Tümünü çalıştır. Biten işler atlanır, yarım kalanlar kaldıkları yerden devam eder.

## 1. Kurulum ve yol kontrolü

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, time
R = '/content/lmc2-task1-w1'
if not os.path.exists(R):
    !git clone -q https://github.com/neuronautsai/lmc2-task1-w1 {R}
!git -C {R} pull -q && git -C {R} log -1 --oneline
!pip -q install segmentation-models-pytorch==0.6.0
%cd {R}
!nvidia-smi --query-gpu=name,memory.total --format=csv

D = '/content/drive/MyDrive/LMC2'
PAIRS = f'{D}/outputs/dup_check/pairs_excluded.csv'
MAN = f'{D}/task1/data_manifest_full.csv'
SHARDS = f'{D}/prepared_shards'  # doğru klasör, outputs/w4/prepared_p2_shards değil
RA = f'{D}/outputs/w5/smp_r34_s42_v2/best.pt'
SIZE = 3000
for p in (PAIRS, MAN, f'{SHARDS}/prepared_shard_00.tar', RA):
    print(os.path.exists(p), p)
    assert os.path.exists(p), f'eksik: {p}'

## 2. Run A: resmi tarza benzer skor (Hann 128) ve 3 model ile TTA süresi

In [ ]:
SA = f'{D}/outputs/w5/score_smp_r34_s42'
if os.path.exists(f'{SA}/official_score_info.json'):
    print('skor zaten var')
    !cat {SA}/summary_official.csv
else:
    !python score_official.py --checkpoint {RA} --window hann --overlap 128 --out {SA} --pairs-csv {PAIRS} --resume
!python time_inference.py --checkpoint {RA} {RA} {RA} --size {SIZE} --tta 1 4 8

## 3. Eğitim verisinin geri yüklenmesi

In [ ]:
if not os.path.exists('/content/prepared'):
    !python prepare_dataset.py restore --shards-dir {SHARDS} --out /content/prepared
print(len(os.listdir('/content/prepared')), 'dosya /content/prepared içinde')

## 4. Base held-out eğitimi, kaldığı yerden devam

Eski oturum hâlâ eğitiyorsa bu hücre kendini atlar.

In [ ]:
H = f'{D}/outputs/w5/base_heldout_s42'
age = (time.time() - os.path.getmtime(f'{H}/last.pt')) / 60 if os.path.exists(f'{H}/last.pt') else None
print('last.pt en son', None if age is None else round(age), 'dakika önce yazılmış')
if age is not None and age < 10:
    print('eski oturum hâlâ eğitiyor görünüyor, burada atlandı')
else:
    !python train.py --out {H} --seed 42 --select-on heldout --prepared-dir /content/prepared --pairs-csv {PAIRS} --resume

## 5. Fokal düzlem testi (madde 1)

In [ ]:
DF = f'{D}/outputs/w5/defocus_smp_r34_s42'
if os.path.exists(f'{DF}/macro_by_offset.csv'):
    print('defocus zaten var')
    !cat {DF}/macro_by_offset.csv
else:
    !python defocus_eval.py --checkpoint {RA} --manifest {MAN} --pairs-csv {PAIRS} --out {DF}

## 6. smp seed 123 ve 456 (Run A ile aynı ayarlar)

In [ ]:
for s in (123, 456):
    !python train.py --out {D}/outputs/w5/smp_r34_s{s} --seed {s} --arch smp --encoder resnet34 --select-on heldout --prepared-dir /content/prepared --pairs-csv {PAIRS} --resume